# Using Open Source LLMs Natively

Here we will see briefly how you can use popular open source LLM APIs including

- Hugging Face Transformers
- Hugging Face Serverless Inference APIs
- Hugging Face Inference Client
- Groq Cloud

## Install Dependencies

In [1]:
!pip install transformers==4.53.2
!pip install accelerate==1.9.0 # useful when using models with GPUs locally via huggingface
!pip install groq==0.30.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 57.2 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.1/367.1 kB 15.5 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 14.0 MB/s eta 0:00:00


## Get Hugging Face Access Token

Here you need to get an access token to be able to download or access models using Hugging Face's platform:

- Hugging Face Access Token: Go [here](https://huggingface.co/settings/tokens) and create a key with write permissions. You need to setup an account which is totally free of cost.


1. Go to [Settings -> Access Tokens](https://huggingface.co/settings/tokens) after creating your account and make sure to create a new access token with write permissions

![](https://i.imgur.com/dtS6tFr.png)

2. Remember to __Save__ your key somewhere safe as it will just be shown once as shown below. So copy and save it in a local secure file to use it later on. If you forget, just create a new key anytime.

![](https://i.imgur.com/NmZmpmw.png)

## Load Hugging Face Access Token


In [2]:
from getpass import getpass

hf_key = getpass("Enter your Hugging Face Access Token: ")

Enter your Hugging Face Access Token: ··········


## Configure Key in Environment


In [3]:
import os

os.environ["HF_TOKEN"] = hf_key

## Using LLMs Locally with Hugging Face

This is if you want to download LLMs locally completely and run it without the need of sending your data to any external server. Do note you would need a GPU to run any of these models as even the smaller language models are still essentially quite big.

Certain LLMs are gated like [Meta Llama 3.2 1B Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) so make sure to apply for access as shown below else you will get an error when using the model

![](https://i.imgur.com/M88MOu5.png)

## Load the LLM locally using Huggingface

In [21]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
import torch

model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda",
    torch_dtype=torch.bfloat16
)

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [22]:
chat = [
    { "role": "user", "content": "Explain what is Generative AI in 2 bullet points" },
]
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 02 Feb 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Explain what is Generative AI in 2 bullet points<|eot_id|><|start_header_id|>assistant<|end_header_id|>




Remember to always refer to the [__documentation__](https://huggingface.co/docs/transformers/v4.18.0/en/main_classes/text_generation#transformers.generation_utils.GenerationMixin.generate) where all the arguments of the generation pipeline are mentioned in detail. Most notably:

- **max_length:** The maximum length of the sequence to be generated
- **max_new_tokens:** The maximum numbers of tokens to generate, ignore the current number of tokens. Use either max_new_tokens or max_length but not both, they serve the same purpose
- **do_sample:** Whether or not to use sampling. False means use greedy decoding i.e temperature=0
- **temperature:** Between 0 - 1, The value used to module the next token probabilities. Higher temperature means the results may vary and be more creative

In [23]:
inputs = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt")
outputs = model.generate(input_ids=inputs.to(model.device), max_new_tokens=1000)
print(tokenizer.decode(outputs[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 02 Feb 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Explain what is Generative AI in 2 bullet points<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here are 2 bullet points explaining Generative AI:

• **Artificial Intelligence that creates new content**: Generative AI is a type of artificial intelligence that can create new content, such as text, images, music, or videos, based on patterns and relationships learned from existing data. This allows machines to generate novel and original ideas, rather than simply reproducing existing ones.

• **Automatically generating new data**: Generative AI can also be used to generate new data that can be used for training, testing, or prediction. For example, a generative model might generate new images of a specific style or subject, which can be used to improve the accuracy of machine learning models.<|eot_id|>


### Pipelines make it easier to send prompts

You don't need to encode and decode your inputs and outputs everytime

In [24]:
llama_pipe = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="cuda",
)

Device set to use cuda


In [25]:
chat = [
    { "role": "user", "content": "Explain what is Generative AI in 2 bullet points" },
]

In [26]:
response = llama_pipe(chat, max_new_tokens=1000)
print(response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[{'generated_text': [{'role': 'user', 'content': 'Explain what is Generative AI in 2 bullet points'}, {'role': 'assistant', 'content': 'Here are two bullet points explaining Generative AI:\n\n• **Creative Generation**: Generative AI is a type of artificial intelligence that can create new, original content, such as art, music, writing, and even entire stories, by generating new variations of existing patterns and structures. This process is often referred to as "generating" or "creating," and it can produce unique and original works that are not limited by the input provided.\n\n• **Automated Output**: Generative AI systems can produce output without human intervention, allowing for the creation of complex and diverse content without the need for manual editing or curation. This can be useful for applications such as content generation, design, and even artistic creation, where the goal is to produce original and innovative works.'}]}]


In [27]:
print(response[0]["generated_text"][-1]['content'])

Here are two bullet points explaining Generative AI:

• **Creative Generation**: Generative AI is a type of artificial intelligence that can create new, original content, such as art, music, writing, and even entire stories, by generating new variations of existing patterns and structures. This process is often referred to as "generating" or "creating," and it can produce unique and original works that are not limited by the input provided.

• **Automated Output**: Generative AI systems can produce output without human intervention, allowing for the creation of complex and diverse content without the need for manual editing or curation. This can be useful for applications such as content generation, design, and even artistic creation, where the goal is to produce original and innovative works.


## Using LLMs via Hugging Face Inference APIs

Thankfully HuggingFace has made its [__Inference API__](https://huggingface.co/docs/api-inference/quicktour) free to use with some basic rate limits etc. in place so you don't end up making unlimited requests on it's servers.

The best part is you can access 150,000+ deep learning models without worrying about your infrastructure.

## Load Hugging Face Access Token


In [5]:
from getpass import getpass

hf_key = getpass("Enter your Hugging Face Access Token: ")

Enter your Hugging Face Access Token: ··········


## Configure Key in Environment


In [6]:
import os

os.environ["HF_TOKEN"] = hf_key

### Create LLM API Access Function

Here we create a basic function which can access any LLM API endpoint available on HuggingFace.

For more details refer to the [detailed documentation](https://huggingface.co/docs/api-inference/detailed_parameters#text-generation-task) as needed.

In [7]:
import requests

headers = {"Authorization": "Bearer "+hf_key}

def query(payload, API_URL):
  response = requests.post(API_URL, headers=headers, json=payload)
  print('API Response:', response)
  return response.json()

## Create LLM API Access Config

Here we decide which LLMs we will access by getting their inference API endpoints.

We also set some general configuration settings. You can find the [detailed documentation](https://huggingface.co/docs/api-inference/detailed_parameters#text-generation-task) here.

Some useful config settings include:

- max_new_tokens: The amount of new tokens to be generated in the response
- do_sample: Whether or not to use sampling. False means use greedy decoding i.e temperature=0
- temperature: Between 0 - 1, The value used to module the next token probabilities. Higher temperature means the results may vary and be more creative
- return_full_text: If set to False, does not return your input prompt to the model
- wait_for_model:  If the model is not ready, wait for it instead of receiving 503. It limits the number of requests required to get your inference done
- repetition_penalty: The more a token is used within generation the more it is penalized to not be picked in successive generation passes.

In [8]:
HF_API_URL = "https://router.huggingface.co/v1/chat/completions" # updated to new endpoint as per HF changes https://huggingface.co/docs/huggingface_hub/v0.13.2/en/guides/inference
# need to mention model provider as per new HF syntax https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct?inference_api=true&inference_provider=novita&language=python&client=requests
model_name = "meta-llama/Llama-3.2-1B-Instruct:novita"
params = {
    "wait_for_model": True,
    "return_full_text": False,
    "max_new_tokens": 1000,
}


In [9]:
prompt =  "Explain what is Generative AI in 2 bullet points"
# updated prompt input format as per https://huggingface.co/docs/huggingface_hub/v0.13.2/en/guides/inference
messages = [
    {
        "role": "user",
        "content": prompt
    }
]

In [11]:
# updated payload format as per https://huggingface.co/docs/huggingface_hub/v0.13.2/en/guides/inference
output = query(payload={
                "messages": messages,
                "parameters": params,
                "model": model_name,
                },
                API_URL=HF_API_URL)

print(output)
print(output['choices'][0]['message']['content'])

API Response: <Response [400]>
{'code': 400, 'message': "[{'type': 'extra_forbidden', 'loc': ('body', 'parameters'), 'msg': 'Extra inputs are not permitted', 'input': {'max_new_tokens': 1000, 'return_full_text': False, 'wait_for_model': True}}]", 'type': 'BadRequestError'}


KeyError: 'choices'

In [12]:
output = query(payload={
    "model": model_name,
    "messages": messages,
    "max_tokens": 1000,
    },
               API_URL=HF_API_URL)
print(output)
print(output['choices'][0]['message']['content'])

API Response: <Response [200]>
{'id': 'chatcmpl-e12d516416be4e3f849f723056857701', 'object': 'chat.completion', 'created': 1770042502, 'model': 'meta-llama/llama-3.2-1b-instruct', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Here are two bullet points explaining what Generative AI is:\n\n• **Automated creation of new content**: Generative AI is a type of artificial intelligence (AI) that uses algorithms and machine learning to create new content, such as text, images, music, and videos, that is similar to but not necessarily identical to the original content. This can be done through various tasks, including text generation, image synthesis, and music composition, where the AI is trained on large datasets and can produce novel outputs that are often surprising or unexpected.\n\n• **Advancements in creative and informative tasks**: Generative AI has the potential to revolutionize various fields, including art, design, writing, and even education, by automating r

## Using LLMs via Hugging Face Inference Client

Thankfully HuggingFace has made its new [__Inference Client__](https://huggingface.co/docs/huggingface_hub/en/package_reference/inference_client) free to use with some basic rate limits etc. in place so you don't end up making unlimited requests on its servers.

The best part is you can access 150,000+ deep learning models without worrying about your infrastructure. Similar to the inference API

In [13]:
from huggingface_hub import InferenceClient

Feel free to refer to the [documentation](https://huggingface.co/docs/huggingface_hub/en/package_reference/inference_client#huggingface_hub.InferenceClient) at any time as needed for more details on function names, arguments and more.

In [14]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"
client = InferenceClient(model=model_name, api_key=hf_key)

chat = [
    { "role": "user", "content": "Explain what is Generative AI in 2 bullet points" },
]

response = client.chat_completion(chat, max_tokens=1000)
print(response)

ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content='Here are two bullet points explaining what Generative AI is:\n\n• **Artificial Intelligence that generates new content**: Generative AI is a type of artificial intelligence (AI) that uses algorithms and machine learning to create new content, such as text, images, music, or videos, that is similar to existing content but with some key differences. It can generate new and original content that is often indistinguishable from the original, making it a valuable tool for applications such as content creation, art generation, and even creative writing.\n\n• **Learning to create new content**: Generative AI learns to generate new content by analyzing large datasets, such as text or images, and identifying patterns and relationships within them. By using these patterns and relationships, the AI can generate new content that is not only simila

In [15]:
print(response.choices[0].message.content)

Here are two bullet points explaining what Generative AI is:

• **Artificial Intelligence that generates new content**: Generative AI is a type of artificial intelligence (AI) that uses algorithms and machine learning to create new content, such as text, images, music, or videos, that is similar to existing content but with some key differences. It can generate new and original content that is often indistinguishable from the original, making it a valuable tool for applications such as content creation, art generation, and even creative writing.

• **Learning to create new content**: Generative AI learns to generate new content by analyzing large datasets, such as text or images, and identifying patterns and relationships within them. By using these patterns and relationships, the AI can generate new content that is not only similar to existing content but also unique and original. This process of learning and generating new content is often referred to as "meta-learning" and is a key 

## Get Grok API

Here you need to get an access token to be able to access models using Grok's platform via APIs:

- Groq API Key: Go [here](https://console.groq.com/keys) and create an API key. You need to setup an account which is totally free of cost. Also while Groq has a generous free tier, there are also paid plans if you are interested.


1. Go to [Groq Cloud -> Create API Key](https://console.groq.com/keys) after creating your account and make sure to create a new API Key as shown

![](https://i.imgur.com/tgHXlcV.png)

2. Remember to __Save__ your key somewhere safe as it will just be shown once as shown below. So copy and save it in a local secure file to use it later on. If you forget, just create a new key anytime.

![](https://i.imgur.com/Q27AgA1.png)

## Load Groq API Credentials


In [16]:
from getpass import getpass

groq_key = getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


## Using Open Source LLMs Directly via Groq API

This is if you want to use it without wrappers like LangChain, we will show you how you use open LLMs like Meta Llama 3.2 Instruct using Groq APIs. The free tier should be good enough for most experiments.

## API Pricing

Right now the best models to use include Mistral, Gemma 2 and Llama 3.1 and 3.2. Check out [pricing details here for free API](https://console.groq.com/settings/limits) and [here for paid API](https://groq.com/pricing/)

![](https://i.imgur.com/JE8lfXV.png)

## Use Groq for Prompting Open Source LLMs

In [17]:
from groq import Groq

groq_client = Groq(api_key=groq_key)

In [18]:
def get_completion_chatgroq(prompt, model="meta-llama/llama-guard-4-12b"):
    messages = [{"role": "user", "content": prompt}]
    response = groq_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, # degree of randomness of the model's output
    )
    return response.choices[0].message.content

In [19]:
# models keep getting updated and older ones get deprecated
# refer to latest list here: https://console.groq.com/dashboard/limits
prompt = 'Explain Generative AI in 2 bullet points'
response = get_completion_chatgroq(prompt=prompt, model="meta-llama/llama-4-scout-17b-16e-instruct")

print(response)

Here are two bullet points explaining Generative AI:

* **What it does:** Generative AI is a type of artificial intelligence that can create new, original content, such as images, videos, music, text, and more. It uses complex algorithms to learn patterns and relationships in data, and then generates new data that is similar in style, structure, and content.
* **How it works:** Generative AI typically uses a technique called deep learning, which involves training a neural network on a large dataset of existing content. The network learns to identify key features and patterns in the data, and then uses this knowledge to generate new content that is likely to be similar. This can be done through various methods, such as Generative Adversarial Networks (GANs) or Variational Autoencoders (VAEs).


In [20]:
prompt = 'Explain Generative AI in 2 bullet points'
response = get_completion_chatgroq(prompt=prompt, model="llama-3.3-70b-versatile")

print(response)

Here are 2 bullet points explaining Generative AI:

* **Definition and Purpose**: Generative AI refers to a type of artificial intelligence that uses machine learning algorithms to generate new, original content, such as images, videos, music, text, or data, that is similar in style and structure to existing data. The purpose of generative AI is to create new, synthetic data that can be used for various applications, such as data augmentation, content creation, and simulation.
* **Key Techniques and Applications**: Generative AI uses techniques such as Generative Adversarial Networks (GANs), Variational Autoencoders (VAEs), and Transformers to learn patterns and relationships in data and generate new content. Applications of generative AI include generating realistic images and videos, creating synthetic data for training machine learning models, composing music and writing text, and simulating real-world environments for testing and training autonomous systems.
